# 05 · UCF-Crime karelerinin resmi olay aralıklarıyla karşılaştırılması
GPU gerekmez, yeni model eğitilmez. `02_tum_deneyler`'in kaydettiği skorları (`skorlar/seed42.npz`) kullanır.

Kullandığımız 7 videonun tamamı UCF-Crime'ın **resmi test listesinde** yer alıyor. Resmi zamansal etiketleri aşağıya elle yazıldı. Dosya adlarındaki sayılar 10'ar artıyor, bu sayılar videonun orijinal kare numarasıdır.

Notebook üç şeyi hesaplar:
1. Her anomali videosunda kaç karenin resmi olay aralığının içinde, kaçının dışında kaldığı.
2. Resmi etiketlerle AUC: normal videoların sekansları ile tamamı olay aralığında kalan sekanslar karşılaştırılır.
3. Aynı video içi AUC: anomali videolarında olay aralığındaki sekanslar ile olay dışındaki sekanslar karşılaştırılır. Bu, UR Fall'daki aynı video içi testin karşılığıdır.

Çıktılar: `revizyon_sonuclari_v2/ek_analizler/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

PROJ    = '/content/drive/MyDrive/AnomalyProject'
FRAMES  = f'{PROJ}/frames'
OUT_DIR = f'{PROJ}/revizyon_sonuclari_v2'
EK      = f'{OUT_DIR}/ek_analizler'; os.makedirs(EK, exist_ok=True)
UCF = {'Normal': 'ucf_normal', 'Kavga (Fighting)': 'ucf_fighting',
       'Hırsızlık (Stealing)': 'ucf_robbery', 'Soygun (Burglary)': 'ucf_accident'}
OLD = {'Kavga (Fighting)': 'Kavga', 'Hırsızlık (Stealing)': 'Hırsızlık', 'Soygun (Burglary)': 'Kaza'}
SEQ_LEN, SEQ_STEP = 8, 2
EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

# Resmi UCF-Crime test etiketleri (Temporal_Anomaly_Annotation_for_Testing_Videos.txt)
# video: (toplam kare, [(olay başı, olay sonu), ...]); normal videolarda olay yok
ANN = {'Normal_Videos_003_x264': (2822, []), 'Normal_Videos_006_x264': (450, []), 'Normal_Videos_010_x264': (1053, []),
       'Fighting003_x264': (3102, [(1820, 3103)]), 'Fighting018_x264': (1389, [(80, 420)]),
       'Stealing019_x264': (4911, [(2730, 2790), (4170, 4350)]), 'Burglary005_x264': (7729, [(4710, 5040)])}

def parse(name):
    m = re.search(r'([A-Za-z]+(?:_Videos_)?\d{3}_x264)_(\d+)\.\w+$', name)
    return (m[1], int(m[2])) if m else (None, None)
def in_event(vid, fr): return any(a <= fr <= b for a, b in ANN[vid][1])

## 1 · Anomali videolarındaki karelerin kaçı gerçekten olay anında?

In [ ]:
F = {}   # kategori → (dosya adları, video, kare no) — 02 notebook'u ile aynı sıralama (alfabetik)
for cat, folder in UCF.items():
    names = sorted(f for f in os.listdir(f'{FRAMES}/{folder}') if f.lower().endswith(EXTS))
    pv = [parse(n) for n in names]
    assert all(v in ANN for v, _ in pv), f'{folder}: tanınmayan video adı {set(v for v, _ in pv if v not in ANN)}'
    F[cat] = (names, [v for v, _ in pv], np.array([f for _, f in pv]))
rows = []
for cat, (names, vids, frs) in F.items():
    for v in dict.fromkeys(vids):
        fr = np.sort(frs[np.array(vids) == v]); ins = np.array([in_event(v, f) for f in fr])
        rows.append({'Kategori': cat, 'Video': v, 'Videonun toplam karesi': ANN[v][0],
                     'Resmi olay aralığı': ', '.join(f'{a}–{b}' for a, b in ANN[v][1]) or '— (normal)',
                     'Kullanılan kare aralığı': f'{fr[0]}–{fr[-1]}', 'Kullanılan kare': len(fr),
                     'Olay içinde': int(ins.sum()) if ANN[v][1] else '—',
                     'Olay dışında': int((~ins).sum()) if ANN[v][1] else '—'})
K = pd.DataFrame(rows); display(K)
an = K[K['Kategori'] != 'Normal']
print(f"Anomali etiketli {an['Kullanılan kare'].sum()} karenin {an['Olay içinde'].sum()} tanesi resmi olay aralığında "
      f"(%{100*an['Olay içinde'].sum()/an['Kullanılan kare'].sum():.1f}).")
K.to_csv(f'{EK}/ucf_resmi_olay_araligi.csv', index=False, encoding='utf-8-sig')

## 2 · Resmi etiketlerle AUC
Sekanslar 02 notebook'undaki gibi kurulur: klasördeki kareler alfabetik sıralanır, 8 karelik pencereler 2'şer kaydırılır. Skorlar yeniden hesaplanmaz, kayıtlı olanlar kullanılır.

Sekans türleri:
- **Olay:** 8 karesinin tamamı aynı anomali videosundan ve resmi olay aralığının içinden.
- **Olay dışı:** 8 karesinin tamamı aynı anomali videosundan ve olay aralığının dışından.
- **Karışık:** Ne olay ne olay dışı olan sekanslar. Bunlar analize alınmaz.

Ayrıca bir sekansın 8 karesinin orijinal videoda ardışık olup olmadığı da sayılır. Alfabetik sıralamada `_1090`'dan sonra `_110` gelir, yani bazı sekanslarda zaman sırası bozulur.

In [ ]:
S0 = np.load(f'{OUT_DIR}/skorlar/seed42.npz', allow_pickle=True)['S'].item()['cross']
for new, old in OLD.items():
    if new not in S0 and old in S0: S0[new] = S0[old]
def win_idx(n): return [list(range(s, s + SEQ_LEN)) for s in range(0, n - SEQ_LEN + 1, SEQ_STEP)]

seq = []
for cat, (names, vids, frs) in F.items():
    idx = win_idx(len(names)); sc = S0[cat]
    assert len(idx) == len(sc), f'{cat}: {len(idx)} pencere ama {len(sc)} skor — sıralama 02 ile uyuşmuyor'
    for w, s in zip(idx, sc):
        vv = {vids[i] for i in w}; f = frs[w]
        if len(vv) > 1: kind = 'iki video karışık'
        elif cat == 'Normal': kind = 'normal'
        else:
            v = vids[w[0]]; ins = [in_event(v, x) for x in f]
            kind = 'olay' if all(ins) else ('olay dışı' if not any(ins) else 'karışık')
        seq.append({'Kategori': cat, 'Video': vids[w[0]], 'Tür': kind, 'Skor': float(s),
                    'Zaman sırası bozuk': bool(len(vv) == 1 and (np.diff(f) != 10).any())})
Q = pd.DataFrame(seq)
display(pd.crosstab([Q['Kategori'], Q['Video']], Q['Tür'], margins=True))
print(f"Zaman sırası bozuk sekans oranı: %{100*Q['Zaman sırası bozuk'].mean():.1f} ({Q['Zaman sırası bozuk'].sum()}/{len(Q)})")

def A(neg, pos):
    a, b = Q[neg]['Skor'].values, Q[pos]['Skor'].values
    return (round(100*roc_auc_score(np.r_[np.zeros(len(a)), np.ones(len(b))], np.r_[a, b]), 2), len(a), len(b)) if len(a) and len(b) else (None, len(a), len(b))
nor, olay, dis = Q['Tür'] == 'normal', Q['Tür'] == 'olay', Q['Tür'] == 'olay dışı'
an_all = (Q['Kategori'] != 'Normal')
R = [('Makaledeki kurulum: normal videolar vs anomali videolarının tüm sekansları', A(nor, an_all)),
     ('Resmi etiket: normal videolar vs yalnızca olay sekansları', A(nor, olay)),
     ('Normal videolar vs anomali videolarının olay DIŞI sekansları', A(nor, dis)),
     ('Aynı video içi: olay dışı vs olay sekansları', A(dis, olay))]
T = pd.DataFrame([{'Karşılaştırma': k, 'AUC (%)': a, 'Negatif sekans': n0, 'Pozitif sekans': n1} for k, (a, n0, n1) in R])
for v in Q.loc[olay, 'Video'].unique():
    m = Q['Video'] == v
    a, n0, n1 = A(m & dis, m & olay); T.loc[len(T)] = [f'Aynı video içi, yalnızca {v}', a, n0, n1]
display(T); T.to_csv(f'{EK}/ucf_resmi_etiket_auc.csv', index=False, encoding='utf-8-sig')

In [ ]:
lines = ['# UCF-Crime · resmi olay aralıklarıyla karşılaştırma', '', K.to_markdown(index=False), '',
         f"Anomali etiketli {an['Kullanılan kare'].sum()} karenin {an['Olay içinde'].sum()} tanesi resmi olay aralığında.", '',
         pd.crosstab(Q['Kategori'], Q['Tür']).to_markdown(), '',
         f"Zaman sırası bozuk sekans oranı: %{100*Q['Zaman sırası bozuk'].mean():.1f}", '',
         T.to_markdown(index=False)]
open(f'{EK}/ucf_resmi_etiket_sayilari.md', 'w', encoding='utf-8').write('\n'.join(lines))
print('\n'.join(lines)); print('\n✅ Kaydedildi →', EK)